In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import os
  
# fetch dataset 
statlog_german_credit_data = fetch_ucirepo(id=144)

# data (as pandas dataframes) 
X = statlog_german_credit_data.data.features 
y = statlog_german_credit_data.data.targets 

# metadata 
print(statlog_german_credit_data.metadata) 
  
# variable information 
print(statlog_german_credit_data.variables)

In [ ]:
column_names = {
    "Attribute1": "checking_status",
    "Attribute2": "duration_months",
    "Attribute3": "credit_history",
    "Attribute4": "purpose",
    "Attribute5": "credit_amount",
    "Attribute6": "savings",
    "Attribute7": "employment_since",
    "Attribute8": "installment_rate",
    "Attribute9": "personal_status_sex",
    "Attribute10": "other_debtors",
    "Attribute11": "residence_since",
    "Attribute12": "property",
    "Attribute13": "age_years",
    "Attribute14": "other_installment_plans",
    "Attribute15": "housing",
    "Attribute16": "existing_credits",
    "Attribute17": "job",
    "Attribute18": "dependents",
    "Attribute19": "telephone",
    "Attribute20": "foreign_worker",
}

X = X.rename(columns=column_names)
y = y.rename(columns={"class": "credit_risk"})

print(X.columns.tolist())
print(y.columns.tolist())

In [ ]:
# Compute fairness measures: male vs female
from sklearn.metrics import confusion_matrix, true_positive_rate

male_codes = ['A91', 'A93', 'A94']
female_codes = ['A92', 'A95']

gender = data['personal_status_sex'].apply(lambda x: 'Male' if x in male_codes else 'Female')

y_true = y.values
y_pred = model_predictions

male_mask = gender == 'Male'
female_mask = gender == 'Female'

y_true_male = y_true[male_mask]
y_pred_male = y_pred[male_mask]

y_true_female = y_true[female_mask]
y_pred_female = y_pred[female_mask]

def calculate_tpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    tpr = tp / (tp + fn)
    return tpr

tpr_male = calculate_tpr(y_true_male, y_pred_male)
tpr_female = calculate_tpr(y_true_female, y_pred_female)

equal_opportunity_diff = abs(tpr_male - tpr_female)
equal_opportunity_ratio = min(tpr_male, tpr_female) / max(tpr_male, tpr_female)

print(f"Male TPR (True Positive Rate):     {tpr_male:.4f}")
print(f"Female TPR (True Positive Rate):   {tpr_female:.4f}")
print(f"\nEqual Opportunity Difference:      {equal_opportunity_diff:.4f}")
print(f"Equal Opportunity Ratio:           {equal_opportunity_ratio:.4f}")
print(f"\nFairness Status:")
if equal_opportunity_diff < 0.1:
    print("✓ FAIR: TPR difference is acceptable (< 0.1)")
else:
    print("✗ UNFAIR: Significant TPR difference between groups")